# Trade Bot - LSTM Training

**2 parties :**
1. Préparation données (~10min) — sauvegarde .npy sur GCS
2. Training d'UN modèle (~40min) — charge .npy, entraîne, sauvegarde

**Si Colab coupe :** relance à partir de la section PARTIE 2 (les données sont déjà sur GCS)

**Pour chaque horizon :** change `HORIZON` et relance la PARTIE 2

---
## CONFIG

In [ ]:
HORIZON = '15m'  # ← Change: '5m', '15m', ou '1h'

In [ ]:
from google.colab import auth
auth.authenticate_user()
!gcloud config set project trade-ml-bot

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, roc_curve
import os

print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')
print(f'Horizon: {HORIZON}')

---
## PARTIE 1 — Préparation données (~10min)

⚠️ **Skip cette partie si les .npy sont déjà sur GCS** (Colab a coupé après la prep)

In [ ]:
# Vérifie si les données préparées existent déjà sur GCS
import subprocess
result = subprocess.run(['gsutil', 'ls', 'gs://trade-ml-bot-data/prepared/features.npy'], capture_output=True, text=True)
if result.returncode == 0:
    print('✅ Données préparées déjà sur GCS — tu peux skip jusqu\'à PARTIE 2')
    SKIP_PREP = True
else:
    print('Données pas encore préparées — lance la suite')
    SKIP_PREP = False

In [ ]:
if not SKIP_PREP:
    !gsutil cp gs://trade-ml-bot-data/dataset_wide_5.gz /tmp/dataset_wide_5.gz
    !gsutil cp gs://trade-ml-bot-data/dataset_wide_5_labels.gz /tmp/dataset_wide_5_labels.gz
    !gsutil cp gs://trade-ml-bot-data/dataset_wide_5_norm.csv.gz /tmp/dataset_wide_5_norm.csv.gz
    print('Download OK')
else:
    print('Skip — déjà préparé')

In [ ]:
if not SKIP_PREP:
    print('Loading dataset...')
    df = pd.read_csv('/tmp/dataset_wide_5.gz', compression='gzip')
    print(f'Dataset: {df.shape}')

    print('Loading labels...')
    labels_df = pd.read_csv('/tmp/dataset_wide_5_labels.gz', compression='gzip')
    print(f'Labels: {labels_df.shape}')

    print('Loading norm stats...')
    norm = pd.read_csv('/tmp/dataset_wide_5_norm.csv.gz', compression='gzip')
    print(f'Norm: {norm.shape}')
else:
    print('Skip — déjà préparé')

In [ ]:
if not SKIP_PREP:
    feature_cols = [c for c in df.columns if c != 'time']
    print(f'{len(feature_cols)} features')

    norm_dict = norm.set_index('feature')
    for col in feature_cols:
        if col in norm_dict.index:
            mean = norm_dict.loc[col, 'mean']
            std = norm_dict.loc[col, 'std']
            if std > 0:
                df[col] = (df[col] - mean) / std

    df[feature_cols] = df[feature_cols].fillna(0)
    print('Normalisation OK')
else:
    print('Skip — déjà préparé')

In [ ]:
if not SKIP_PREP:
    df['time'] = pd.to_datetime(df['time'])
    labels_df['time'] = pd.to_datetime(labels_df['time'])

    merged = df.merge(labels_df[['time', 'label_5m', 'label_15m', 'label_1h']], on='time', how='inner')
    merged = merged.sort_values('time').reset_index(drop=True)
    merged = merged.dropna(subset=['label_5m', 'label_15m', 'label_1h'])

    features_np = merged[feature_cols].values.astype(np.float32)

    # Diagnostic normalisation
    means = np.mean(features_np[:1000], axis=0)
    bad = []
    for i, col in enumerate(feature_cols):
        if abs(means[i]) > 2:
            bad.append(f'  {col}: mean={means[i]:.2f}')

    if bad:
        print('⚠️ Features mal normalisées:')
        print('\n'.join(bad))
        print('\n→ STOP: corriger la table norm')
    else:
        print('✅ Normalisation OK')

    print(f'Merged: {merged.shape}')
    print(f'Période: {merged["time"].min()} → {merged["time"].max()}')
else:
    print('Skip — déjà préparé')

In [ ]:
if not SKIP_PREP:
    lab_5m = merged['label_5m'].values.astype(np.float32)
    lab_15m = merged['label_15m'].values.astype(np.float32)
    lab_1h = merged['label_1h'].values.astype(np.float32)

    os.makedirs('/tmp/prepared', exist_ok=True)
    np.save('/tmp/prepared/features.npy', features_np)
    np.save('/tmp/prepared/lab_5m.npy', lab_5m)
    np.save('/tmp/prepared/lab_15m.npy', lab_15m)
    np.save('/tmp/prepared/lab_1h.npy', lab_1h)
    np.save('/tmp/prepared/feature_cols.npy', np.array(feature_cols))

    !gsutil -m cp /tmp/prepared/*.npy gs://trade-ml-bot-data/prepared/

    print(f'\n✅ Données préparées sauvegardées sur GCS')
    print(f'features: {features_np.shape}')
else:
    print('Skip — déjà préparé')

---
## PARTIE 2 — Training LSTM (~40min)

Charge les .npy depuis GCS et entraîne le modèle pour l'horizon choisi.

**Si Colab a coupé pendant PARTIE 1 :** relance depuis CONFIG puis ici directement.

In [ ]:
os.makedirs('/tmp/prepared', exist_ok=True)
!gsutil -m cp gs://trade-ml-bot-data/prepared/*.npy /tmp/prepared/

features_np = np.load('/tmp/prepared/features.npy')
labels = np.load(f'/tmp/prepared/lab_{HORIZON}.npy')
feature_cols = np.load('/tmp/prepared/feature_cols.npy', allow_pickle=True)

n_features = features_np.shape[1]
print(f'Features: {features_np.shape}')
print(f'Labels {HORIZON}: {labels.shape}')
print(f'Distribution: {dict(zip(*np.unique(labels, return_counts=True)))}')

In [ ]:
WINDOW_SIZE = 30
BATCH_SIZE = 512

n_samples = len(features_np) - WINDOW_SIZE + 1

train_end = int(n_samples * 0.70)
val_end = int(n_samples * 0.85)

def make_dataset(features, labs, start, end, batch_size, shuffle=False):
    feat_slice = features[start:end + WINDOW_SIZE - 1]
    lab_slice = labs[start + WINDOW_SIZE - 1:end + WINDOW_SIZE - 1]
    ds = tf.keras.utils.timeseries_dataset_from_array(
        data=feat_slice,
        targets=lab_slice,
        sequence_length=WINDOW_SIZE,
        batch_size=batch_size,
        shuffle=shuffle,
    )
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(features_np, labels, 0, train_end, BATCH_SIZE, shuffle=True)
val_ds = make_dataset(features_np, labels, train_end, val_end, BATCH_SIZE)
test_ds = make_dataset(features_np, labels, val_end, n_samples, BATCH_SIZE)

for x, y in train_ds.take(1):
    print(f'Batch check — X: {x.shape}, y: {y.shape}')

print(f'\nTrain: {train_end:,} — Val: {val_end-train_end:,} — Test: {n_samples-val_end:,}')

In [ ]:
EPOCHS = 20
CHECKPOINT_DIR = f'/tmp/checkpoints/lstm_{HORIZON}'
GCS_CHECKPOINT = f'gs://trade-ml-bot-data/checkpoints/lstm_{HORIZON}'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

model = keras.Sequential([
    layers.Input(shape=(WINDOW_SIZE, n_features)),
    layers.LSTM(64, return_sequences=True),
    layers.Dropout(0.5),
    layers.LSTM(32),
    layers.Dropout(0.5),
    layers.Dense(16, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
], name=f'lstm_{HORIZON}')

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc')]
)

# Reprendre depuis un checkpoint GCS si il existe
import subprocess, re
check = subprocess.run(['gsutil', 'ls', f'{GCS_CHECKPOINT}/'], capture_output=True, text=True)
if check.returncode == 0 and '.weights.h5' in check.stdout:
    print('🔄 Checkpoint trouvé sur GCS, téléchargement...')
    os.system(f'gsutil -m cp {GCS_CHECKPOINT}/* {CHECKPOINT_DIR}/')
    # Trouver le dernier epoch
    files = [f for f in os.listdir(CHECKPOINT_DIR) if f.endswith('.weights.h5')]
    if files:
        files.sort()
        latest_file = files[-1]
        model.load_weights(f'{CHECKPOINT_DIR}/{latest_file}')
        match = re.search(r'ep(\d+)', latest_file)
        initial_epoch = int(match.group(1)) if match else 0
        print(f'✅ Reprise à l\'epoch {initial_epoch + 1}/{EPOCHS}')
    else:
        initial_epoch = 0
        print('⚠️ Pas de checkpoint valide, démarrage de zéro')
else:
    initial_epoch = 0
    print('Pas de checkpoint, démarrage de zéro')

if initial_epoch >= EPOCHS:
    print(f'Training déjà terminé ({initial_epoch} epochs). Skip.')
else:
    model.summary()

    # Callback qui upload sur GCS après chaque epoch
    class GCSCheckpoint(keras.callbacks.Callback):
        def on_epoch_end(self, epoch, logs=None):
            path = f'{CHECKPOINT_DIR}/ep{epoch + 1:02d}.weights.h5'
            model.save_weights(path)
            os.system(f'gsutil cp {path} {GCS_CHECKPOINT}/ep{epoch + 1:02d}.weights.h5')
            print(f'  → Checkpoint epoch {epoch + 1} sauvegardé sur GCS')

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor='val_auc', patience=5, mode='max', restore_best_weights=True
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6
        ),
        GCSCheckpoint()
    ]

    print(f'\nTraining LSTM {HORIZON} — {train_end:,} samples, epochs {initial_epoch + 1}→{EPOCHS}')
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        initial_epoch=initial_epoch,
        callbacks=callbacks,
        verbose=1
    )

## Évaluation

In [ ]:
test_loss, test_acc, test_auc = model.evaluate(test_ds, verbose=0)
print(f'Test {HORIZON} — Loss: {test_loss:.4f}, Accuracy: {test_acc:.4f}, AUC: {test_auc:.4f}')

y_test = np.concatenate([y for _, y in test_ds])
y_pred_prob = model.predict(test_ds).flatten()
y_pred = (y_pred_prob > 0.5).astype(int)

print(f'\nClassification Report {HORIZON}:')
print(classification_report(y_test, y_pred, target_names=['Baisse', 'Hausse']))

print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history.history['loss'], label='train')
axes[0].plot(history.history['val_loss'], label='val')
axes[0].set_title('Loss')
axes[0].legend()

axes[1].plot(history.history['accuracy'], label='train')
axes[1].plot(history.history['val_accuracy'], label='val')
axes[1].set_title('Accuracy')
axes[1].legend()

axes[2].plot(history.history['auc'], label='train')
axes[2].plot(history.history['val_auc'], label='val')
axes[2].set_title('AUC')
axes[2].legend()

plt.suptitle(f'LSTM {HORIZON}')
plt.tight_layout()
plt.show()

fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, label=f'AUC = {test_auc:.3f}')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title(f'ROC Curve - LSTM {HORIZON}')
plt.legend()
plt.show()

## Sauvegarde sur GCS

In [ ]:
# Keras format
model.save(f'/tmp/lstm_{HORIZON}.keras')
!gsutil cp /tmp/lstm_{HORIZON}.keras gs://trade-ml-bot-data/models/lstm_{HORIZON}.keras

# TF.js format
!pip install tensorflowjs -q
import tensorflowjs as tfjs
os.makedirs(f'/tmp/models/lstm_{HORIZON}', exist_ok=True)
tfjs.converters.save_keras_model(model, f'/tmp/models/lstm_{HORIZON}')
!gsutil -m cp -r /tmp/models/lstm_{HORIZON}/* gs://trade-ml-bot-data/models/lstm_{HORIZON}/

print(f'\n✅ Modèle LSTM {HORIZON} sauvegardé sur GCS')
!gsutil ls gs://trade-ml-bot-data/models/lstm_{HORIZON}/